In [ ]:
"""
VQ-VAE Performance Analysis: Reconstruction Loss vs Token Configuration
========================================================================
Plots reconstruction loss (L1 in mm) against token sequence length,
with different colors for codebook sizes. Solid lines = seen (vision datasets),
dashed lines = unseen (mmWave datasets).
"""

import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from cycler import cycler

# Plot style configuration
PALETTE = ['#1e90ff', '#ffbb00', '#ff5080', '#a7426d', '#ff3c10', "#282828"]
MARKERS = ['o', 'P', '^', 's', 'p', 'h']

plt.rcParams.update({
    "figure.figsize": [7.0, 4.5],
    "figure.dpi": 100,
    "font.size": 16,
    "font.family": "Arial",
    "lines.linewidth": 2.5,
    "lines.markersize": 8,
    "lines.markeredgewidth": 3,
    "legend.fontsize": "medium",
    "legend.frameon": False,
    "grid.linestyle": "--",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PALETTE) + cycler(marker=MARKERS)

In [ ]:
def load_vqvae_results(work_dirs_path: str = "../../../logs/vqvae") -> pd.DataFrame:
    """
    Load VQ-VAE results from work_dirs/vqvae_tokennum*_tokenclass* directories.
    
    Returns DataFrame with columns:
        - token_num: number of latent tokens
        - token_class: codebook size
        - seen_mse: validation MSE on training datasets (AMASS)
        - unseen_mse: test MSE on MMBody dataset
    """
    work_dirs = Path(work_dirs_path)
    pattern = re.compile(r"vqvae_tokennum(\d+)_tokenclass(\d+)")
    data = []
    
    for dir_path in sorted(work_dirs.iterdir()):
        match = pattern.match(dir_path.name)
        if not match:
            continue
        
        token_num, token_class = int(match.group(1)), int(match.group(2))
        
        # Load seen MSE from training_results.json
        train_file = dir_path / "training_results.json"
        seen_mse = None
        if train_file.exists():
            with open(train_file) as f:
                seen_mse = json.load(f).get('best_metrics', {}).get('validation', {}).get('recon_loss')
        
        # Load unseen MSE from testing_results.json
        test_file = dir_path / "testing_results.json"
        unseen_mse = None
        if test_file.exists():
            with open(test_file) as f:
                unseen_mse = json.load(f).get('mmbody_evaluation', {}).get('avg_losses', {}).get('recon_loss')
        
        if seen_mse is not None and unseen_mse is not None:
            data.append({
                'token_num': token_num,
                'token_class': token_class,
                'seen_mse': seen_mse,
                'unseen_mse': unseen_mse
            })
    
    return pd.DataFrame(data)

# Load data and convert MSE to L1 loss in millimeters
df = load_vqvae_results()
df["seen_l1_mm"] = np.sqrt(df["seen_mse"]) * 1000
df["unseen_l1_mm"] = np.sqrt(df["unseen_mse"]) * 1000
df = df.sort_values(["token_class", "token_num"])

print(f"Loaded {len(df)} model configurations:")
print(df[["token_num", "token_class", "seen_l1_mm", "unseen_l1_mm"]].to_string(index=False))

In [ ]:
# Plot reconstruction loss vs token sequence length
fig, ax = plt.subplots()

# Plot each codebook size with solid (seen) and dashed (unseen) lines
for c in sorted(df["token_class"].unique()):
    sub = df[df["token_class"] == c].sort_values("token_num")
    line, = ax.plot(sub["token_num"], sub["seen_l1_mm"])
    ax.plot(sub["token_num"], sub["unseen_l1_mm"],
            linestyle="--", color=line.get_color(), marker=line.get_marker())

# Axis configuration
ax.set_xscale("log", base=2)
ax.set_xlabel("Latent token sequence length")
ax.set_ylabel("L1 reconstruction loss (mm)")
ax.set_yticks([10, 20, 30, 40])
ax.grid(True, linestyle="--", alpha=0.6, which="both")

# Build legends
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
class_handles = [
    Line2D([0], [0], color=colors[i], marker=MARKERS[i], linestyle='-', linewidth=2.5, markersize=8)
    for i, c in enumerate(sorted(df["token_class"].unique()))
]
class_labels = [f"#Codebook {c}" for c in sorted(df["token_class"].unique())]

style_handles = [
    Line2D([0], [0], color="black", linestyle='-', linewidth=2.5),
    Line2D([0], [0], color="black", linestyle='--', linewidth=2.5),
]

# Add legends: codebook size at top, line style below
fig.legend(class_handles, class_labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.95))
ax.legend(style_handles, ["Vision datasets", "Unseen mmWave datasets"],
          loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.2))

plt.subplots_adjust(top=0.75)
plt.savefig("reconstruction_loss_vs_token_num.pdf", bbox_inches='tight')
plt.show()
